# 06 — Solar Forecasting

Materialises the dimension + bronze → silver → gold tables in
[`../specifications/06-solar-forecasting.md`](../specifications/06-solar-forecasting.md).

**Depends on 04** — reads `volume_forecast_silver_btm_pv` for the behind-the-meter reconciliation so utility-scale
solar (here) and BTM PV (netted into demand in 04) are never double-counted.

**Capability tables**
- `volume_forecast_dim_solar_assets` (solar fleet registry, owned here)
- `volume_forecast_bronze_nwp_solar`, `volume_forecast_bronze_solar_scada`
- `volume_forecast_silver_solar_forecast` (clear-sky + probabilistic P10/P50/P90 per asset × interval)
- `volume_forecast_gold_solar_summary` (midday peak, BTM reconciliation, negative-price risk)

**UC comments:** [`uc_table_comments.py`](./uc_table_comments.py) — applied in the final cell.

In [ ]:
import os
import math
import random
import datetime as dt

from pyspark.sql import functions as F
from pyspark.sql import Row

dbutils.widgets.text("catalog", os.environ.get("DEMO_UC_CATALOG", "energy_utilities"))
dbutils.widgets.text("schema", os.environ.get("DEMO_UC_SCHEMA", "energy_trading2"))

CATALOG = dbutils.widgets.get("catalog").strip() or "energy_utilities"
SCHEMA = dbutils.widgets.get("schema").strip() or "energy_trading2"
print(f"Target: {CATALOG}.{SCHEMA}")
spark.sql(f"USE `{CATALOG}`.`{SCHEMA}`")


def fq(name: str) -> str:
    return f"`{CATALOG}`.`{SCHEMA}`.`{name}`"


random.seed(6)

TODAY = dt.date.today()
# Rolling horizon: 2 history days, today, 2 forecast days (matches 04 / 01 / 03 / 05).
DELIVERY_DATES = [TODAY + dt.timedelta(days=d) for d in (-2, -1, 0, 1, 2)]
LATEST_DATE = DELIVERY_DATES[-1]
N_INTERVALS = 96
NOW_INDEX = 56
ZONES = ["DE", "NL", "FR", "BE", "AT"]
SNAP_TS = dt.datetime.combine(TODAY, dt.time(15, 0))


def interval_ts(day: dt.date, idx: int) -> dt.datetime:
    return dt.datetime.combine(day, dt.time(0, 0)) + dt.timedelta(minutes=15 * idx)


def is_settled(day: dt.date, idx: int) -> bool:
    if day < TODAY:
        return True
    if day == TODAY:
        return idx < NOW_INDEX
    return False


def clear_sky_factor(idx: int) -> float:
    h = idx * 15 / 60.0
    if h <= 6 or h >= 19:
        return 0.0
    return max(0.0, math.sin(math.pi * (h - 6.0) / 13.0))


# ---- Dimension: solar fleet registry (owned here) ----
# (asset_id, name, zone, nameplate_mw, tilt, azimuth, tracking)
SOLAR_ASSETS = [
    ("SOLAR_DE_001", "Bavaria Solar Park",     "DE", 250.0, 30, 180, "FIXED"),
    ("SOLAR_DE_002", "Saxony Tracker Farm",    "DE", 180.0,  0, 180, "SINGLE_AXIS"),
    ("SOLAR_FR_001", "Provence Solar",         "FR", 200.0, 28, 180, "FIXED"),
    ("SOLAR_NL_001", "Groningen Solar",        "NL", 120.0, 35, 180, "FIXED"),
    ("SOLAR_BE_001", "Wallonia Solar",         "BE",  80.0, 32, 180, "FIXED"),
]
asset_rows = [Row(asset_id=a[0], asset_name=a[1], zone_code=a[2], nameplate_mw=a[3],
                  tilt_deg=a[4], azimuth_deg=a[5], tracking=a[6]) for a in SOLAR_ASSETS]
spark.createDataFrame(asset_rows).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(fq("volume_forecast_dim_solar_assets"))

# Stable per-(day, zone) cloud regime.
_cloud_base = {(day, z): random.uniform(10.0, 65.0) for day in DELIVERY_DATES for z in ZONES}


def cloud_cover(day: dt.date, zone: str, idx: int) -> float:
    return max(0.0, min(100.0, _cloud_base[(day, zone)] + random.gauss(0, 10)))

In [ ]:
# ---- Bronze: irradiance / cloud forecast (per zone) ----
nwp_rows = []
for day in DELIVERY_DATES:
    fts = dt.datetime.combine(day, dt.time(6, 0))
    for z in ZONES:
        for idx in range(N_INTERVALS):
            cs = clear_sky_factor(idx)
            clear_ghi = cs * 950.0
            cloud = cloud_cover(day, z, idx) if cs > 0 else cloud_cover(day, z, idx)
            ghi = clear_ghi * (1 - cloud / 100.0 * 0.7)
            nwp_rows.append(Row(
                ingestion_ts=fts, forecast_ts=fts,
                source=random.choice(["ECMWF", "SATELLITE_NOWCAST", "ICON"]),
                zone_code=z, interval_start=interval_ts(day, idx),
                ghi_wm2=round(max(0.0, ghi), 1), cloud_cover_pct=round(cloud, 1),
                clear_sky_ghi_wm2=round(clear_ghi, 1),
            ))
spark.createDataFrame(nwp_rows).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(fq("volume_forecast_bronze_nwp_solar"))

# ---- Bronze: inverter SCADA + Silver: probabilistic generation ----
scada_rows, fc_rows = [], []
for a in SOLAR_ASSETS:
    asset_id, _, zone, nameplate, _, _, tracking = a
    track_gain = 1.12 if tracking == "SINGLE_AXIS" else (1.20 if tracking == "DUAL_AXIS" else 1.0)
    for day in DELIVERY_DATES:
        fts = dt.datetime.combine(day, dt.time(6, 0))
        for idx in range(N_INTERVALS):
            cs = clear_sky_factor(idx)
            clear_mw = min(nameplate, nameplate * cs * track_gain)
            cloud = cloud_cover(day, zone, idx)
            derate = 1 - cloud / 100.0 * 0.7
            p50 = clear_mw * derate
            spread = clear_mw * (0.05 + 0.20 * (cloud / 100.0))  # cloudier = more uncertain
            p10 = max(0.0, p50 - spread)
            p90 = min(nameplate, p50 + spread)
            prev = max(0.0, min(nameplate, p50 * (1 + random.gauss(0, 0.08))))
            settled = is_settled(day, idx)
            actual = max(0.0, min(nameplate, p50 * (1 + random.gauss(0, 0.05)))) if settled else None
            start = interval_ts(day, idx)
            if settled and cs > 0:
                status = "OUTAGE" if random.random() < 0.01 else ("CURTAILED" if (cs > 0.85 and random.random() < 0.05) else "RUNNING")
                if status == "OUTAGE":
                    actual = 0.0
                scada_rows.append(Row(
                    ingestion_ts=start, telemetry_ts=start, asset_id=asset_id,
                    actual_mw=round(actual, 2) if actual is not None else 0.0,
                    panel_temp_c=round(15 + 25 * cs + random.gauss(0, 2), 1),
                    availability_pct=round(0.0 if status == "OUTAGE" else 100.0, 1),
                    status=status,
                ))
            fc_rows.append(Row(
                delivery_date=day, interval_start=start, forecast_ts=fts,
                asset_id=asset_id, zone_code=zone,
                clear_sky_mw=round(clear_mw, 2),
                p10_mw=round(p10, 2), p50_mw=round(p50, 2), p90_mw=round(p90, 2),
                prev_p50_mw=round(prev, 2), forecast_delta_mw=round(p50 - prev, 2),
                actual_mw=round(actual, 2) if actual is not None else None,
            ))
spark.createDataFrame(scada_rows).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(fq("volume_forecast_bronze_solar_scada"))
spark.createDataFrame(fc_rows).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(fq("volume_forecast_silver_solar_forecast"))

print("nwp_solar:", spark.table(fq("volume_forecast_bronze_nwp_solar")).count())
print("solar_scada:", spark.table(fq("volume_forecast_bronze_solar_scada")).count())
print("solar_forecast:", spark.table(fq("volume_forecast_silver_solar_forecast")).count())

In [ ]:
# ---- Gold: solar headline KPIs per zone per day (with BTM reconciliation from 04) ----
fc = spark.table(fq("volume_forecast_silver_solar_forecast"))

zint = fc.groupBy("delivery_date", "zone_code", "interval_start").agg(
    F.sum("p50_mw").alias("z_p50"),
    F.sum(F.col("p90_mw") - F.col("p10_mw")).alias("z_band"),
)
agg = zint.groupBy("delivery_date", "zone_code").agg(
    F.round(F.avg("z_p50"), 2).alias("total_p50_mw"),
    F.round(F.max("z_p50"), 2).alias("midday_peak_mw"),
    F.round(F.avg("z_band"), 2).alias("band_width_mw"),
)

# BTM PV netted in 04 — surfaced here only for reconciliation, not re-added to supply.
if spark.catalog.tableExists(fq("volume_forecast_silver_btm_pv")):
    btm = (spark.table(fq("volume_forecast_silver_btm_pv"))
           .groupBy("delivery_date", "zone_code")
           .agg(F.round(F.avg("btm_pv_mw"), 2).alias("btm_pv_mw")))
else:
    btm = agg.select("delivery_date", "zone_code").withColumn("btm_pv_mw", F.lit(0.0))

summary = (agg.join(btm, ["delivery_date", "zone_code"], "left")
    .fillna({"btm_pv_mw": 0.0})
    .withColumn("snapshot_ts", F.lit(SNAP_TS).cast("timestamp"))
    .withColumn("negative_price_risk", F.when(F.col("midday_peak_mw") > 250, F.lit("HIGH"))
                                         .when(F.col("midday_peak_mw") > 120, F.lit("MEDIUM")).otherwise(F.lit("LOW")))
    .withColumn("headline", F.when(F.col("negative_price_risk") == "HIGH", F.lit("Midday solar surplus — negative-price / curtailment risk"))
                             .otherwise(F.lit("Solar generation within normal band")))
    .select("delivery_date", "zone_code", "snapshot_ts", "headline",
            "total_p50_mw", "midday_peak_mw", "band_width_mw", "btm_pv_mw", "negative_price_risk"))

summary.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(fq("volume_forecast_gold_solar_summary"))
display(spark.table(fq("volume_forecast_gold_solar_summary")).orderBy(F.col("delivery_date").desc(), "zone_code"))

In [ ]:
# Row counts + Unity Catalog comments.
for t in [
    "volume_forecast_dim_solar_assets",
    "volume_forecast_bronze_nwp_solar",
    "volume_forecast_bronze_solar_scada",
    "volume_forecast_silver_solar_forecast",
    "volume_forecast_gold_solar_summary",
]:
    print(f"  {t:46s}  {spark.table(fq(t)).count():>10,} rows")

from pathlib import Path

_uc_paths = []
try:
    _nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    _uc_paths.append(Path(_nb).parent / "uc_table_comments.py")
except Exception:
    pass
_uc_paths.append(Path.cwd() / "uc_table_comments.py")

_uc_py = next((p for p in _uc_paths if p.is_file()), None)
if _uc_py is None:
    raise FileNotFoundError("uc_table_comments.py not found next to this notebook.")

exec(_uc_py.read_text(), globals())
apply_volume_forecast_notebook_06_comments(spark, CATALOG, SCHEMA)
print("UC comments applied for notebook 06.")